# Rapid Response — Detektör Eğitimi (Colab)
9-sınıf YOLO11 detektörünü (`best_model.pt`) eğitir.
**ÖNCE:** veriyi Drive'a `sources.yaml` yapısında koy (bkz. HANDOFF.md Bölüm 4).
Çalıştırma: **Runtime → Change runtime type → GPU (T4)**, sonra hücreleri sırayla çalıştır.


## 1) GPU kontrolü


In [1]:
!nvidia-smi


Fri Jun 26 13:10:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2) Drive'ı bağla


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3) Repo + kütüphaneler


In [12]:
!git clone https://${GH_TOKEN}@github.com/f-tsakir/Rapid_Response.git
%cd Rapid_Response
!pip -q install ultralytics pyyaml


Cloning into 'Rapid_Response'...
remote: Enumerating objects: 228, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 228 (delta 0), reused 2 (delta 0), pack-reused 218 (from 1)
Receiving objects: 100% (228/228), 17.12 MiB | 17.82 MiB/s, done.
Resolving deltas: 100% (83/83), done.
/content/drive/MyDrive/dataset/coco/Rapid_Response


## 4) Veri yollarını DOĞRULA
`sources.yaml`'daki her kaynak Drive'da var mı? ❌ olanları ya yükle ya o satırı yorum yap.


In [13]:
import yaml, os
srcs = yaml.safe_load(open('training/sources.yaml'))
for s in srcs:
    p = s.get('images','')
    print(('OK ' if os.path.isdir(p) else 'YOK'), s['name'], '->', p)


OK  tr_plates -> /content/drive/MyDrive/dataset/plaka/tr_plates/train/images
OK  abnormal_behaviour -> /content/drive/MyDrive/dataset/sofor_eylemi/abnormal_behaviour/train/images
YOK mio_tcd -> /content/mio_yolo/images
OK  coco_objects -> /content/drive/MyDrive/dataset/coco/val2017
YOK teknocan_committee -> /content/teknocan_full/images
YOK bilgisayar_committee -> /content/bilgisayar_full/images
YOK detrac_arac -> /content/detrac_arac/images/train


In [14]:
!ls "/content/drive/MyDrive/dataset/nesne_yolcu" 2>/dev/null

bilgisayar_full.zip  teknocan_full.zip


In [15]:
!unzip -q "/content/drive/MyDrive/dataset/nesne_yolcu/teknocan_full.zip" -d /content/
!unzip -q "/content/drive/MyDrive/dataset/nesne_yolcu/bilgisayar_full.zip" -d /content/
!echo "--- teknocan_full ---"; ls /content/teknocan_full
!echo "--- bilgisayar_full ---"; ls /content/bilgisayar_full

--- teknocan_full ---
images	labels
--- bilgisayar_full ---
images	labels


In [16]:
import yaml, os
srcs = yaml.safe_load(open('training/sources.yaml'))
keep = [s for s in srcs if os.path.isdir(s.get('images',''))]
print('Kullanılacak:', [s['name'] for s in keep])
print('Atlanan     :', [s['name'] for s in srcs if s not in keep])
yaml.safe_dump(keep, open('training/sources.yaml','w'), allow_unicode=True)

Kullanılacak: ['tr_plates', 'abnormal_behaviour', 'coco_objects', 'teknocan_committee', 'bilgisayar_committee']
Atlanan     : ['mio_tcd', 'detrac_arac']


## 5) 9-sınıf birleşik set üret


In [10]:
import os
os.makedirs('/content/drive/MyDrive/dataset/coco', exist_ok=True)
%cd /content/drive/MyDrive/dataset/coco
!wget -q http://images.cocodataset.org/zips/val2017.zip && unzip -q val2017.zip && rm val2017.zip
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip && unzip -q annotations_trainval2017.zip && rm annotations_trainval2017.zip
print("COCO Drive'da hazır ✅")

/content/drive/MyDrive/dataset/coco
COCO Drive'da hazır ✅


In [8]:
import yaml, os
srcs = yaml.safe_load(open('training/sources.yaml'))
for s in srcs:
    p = s.get('images','')
    print(('OK ' if os.path.isdir(p) else 'YOK'), s['name'], '->', p)


b.dogan  Dockerfile  main.py  PROJE_DURUMU.md  requirements.txt  training
config	 live	     mobile   README.md        src		 weights


In [3]:
%cd /content/Rapid_Response
!python training/prepare_dataset.py \
  --sources training/sources.yaml --data-yaml training/data.yaml \
  --out /content/datasets/rapid_response --split 0.70 0.15 0.15 --clean \
  --max-per-source 600

/content/Rapid_Response
Traceback (most recent call last):
  File "/content/Rapid_Response/training/prepare_dataset.py", line 235, in <module>
    raise SystemExit(main())
                     ^^^^^^
  File "/content/Rapid_Response/training/prepare_dataset.py", line 186, in main
    items = collect_source(src, name2id, args.max_per_source)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Rapid_Response/training/prepare_dataset.py", line 135, in collect_source
    return read_yolo_source(src, name2id, cap)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Rapid_Response/training/prepare_dataset.py", line 83, in read_yolo_source
    for line in lbl.read_text().splitlines():
                ^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/pathlib.py", line 1028, in read_text
    return f.read()
           ^^^^^^^^
  File "<frozen codecs>", line 319, in decode
KeyboardInterrupt
^C


In [4]:
%cd /content
!wget -q http://images.cocodataset.org/zips/val2017.zip && unzip -q val2017.zip
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip && unzip -q annotations_trainval2017.zip
print("COCO yerelde ✅")

/content
COCO yerelde ✅


In [6]:
import yaml
p='/content/Rapid_Response/training/sources.yaml'
srcs=yaml.safe_load(open(p)); fast=[]
for s in srcs:
    if s['name']=='coco_objects':
        s['images']='/content/val2017'; s['ann']='/content/annotations/instances_val2017.json'; fast.append(s)
    elif s['name'] in ('teknocan_committee','bilgisayar_committee'):
        fast.append(s)
yaml.safe_dump(fast, open(p,'w'), allow_unicode=True)
print('Kullanılacak (hepsi yerel):', [s['name'] for s in fast])

Kullanılacak (hepsi yerel): ['coco_objects', 'teknocan_committee', 'bilgisayar_committee']


In [7]:
%cd /content/Rapid_Response
!python training/prepare_dataset.py --sources training/sources.yaml --data-yaml training/data.yaml \
  --out /content/datasets/rapid_response --split 0.70 0.15 0.15 --clean

/content/Rapid_Response

=== KAYNAK OZETI (goruntu) ===
  coco_objects                   3305  (2313/495/497)
  teknocan_committee              108  (75/16/17)
  bilgisayar_committee            236  (165/35/36)

=== SPLIT (goruntu) ===
  train  2553
  val    546
  test   550

=== SINIF HISTOGRAMI (kutu) ===
   0 arac                  2976
   1 plaka                  324
   2 teknocan               324
   3 bilgisayar             547
   4 telefon                370
   5 sigara                   0  <-- BOS!
   6 sise                  1924
   7 emniyet_kemeri           0  <-- BOS!
   8 kisi                 11454

Cikti: /content/datasets/rapid_response  (data.yaml: training/data.yaml)


## 6) (Opsiyonel) Hızlı test için epoch'u düşür
İlk denemede config'te `training.epochs`'u 100 yerine 20-30 yap; çalıştığını görünce tama çıkar.


In [9]:
# import yaml; c=yaml.safe_load(open('config/config.yaml'));
# c['training']['epochs']=30; yaml.safe_dump(c, open('config/config.yaml','w'))


In [12]:
import yaml
d=yaml.safe_load(open('training/data.yaml'))
d['path']='/content/datasets/rapid_response'
yaml.safe_dump(d, open('training/data.yaml','w'), allow_unicode=True)
print('path =', d['path'])

path = /content/datasets/rapid_response


In [13]:
%cd /content/Rapid_Response
!python training/train_yolo.py --config config/config.yaml --no-synthetic

/content/Rapid_Response
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[train] pretrained=yolo11s.pt data=training/data.yaml synthetic=OFF
New https://pypi.org/project/ultralytics/8.4.79 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.78 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=training/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropo

In [14]:
!cp /content/Rapid_Response/weights/best_model.pt "/content/drive/MyDrive/best_model_7class.pt"
print("✅ best_model_7class.pt Drive'a kaydedildi")

✅ best_model_7class.pt Drive'a kaydedildi


In [15]:
!ls -lah "/content/drive/MyDrive/best_model_7class.pt"

-rw------- 1 root root 19M Jun 26 15:48 /content/drive/MyDrive/best_model_7class.pt


In [16]:
from google.colab import files
files.download("/content/drive/MyDrive/best_model_7class.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import os
print("=== Yerel veriler (session SIFIRLANDI mı?) ===")
for p in ['/content/Rapid_Response','/content/val2017','/content/teknocan_full',
          '/content/bilgisayar_full','/content/datasets/rapid_response']:
    print('OK ' if os.path.exists(p) else 'YOK', p)
print("\n=== Drive dataset yapısı ===")
d='/content/drive/MyDrive/dataset'
print('dataset içi:', os.listdir(d) if os.path.isdir(d) else 'dataset YOK')
s=d+'/sofor_eylemi'
print('sofor_eylemi içi:', os.listdir(s) if os.path.isdir(s) else 'sofor_eylemi YOK')

=== Yerel veriler (session SIFIRLANDI mı?) ===
YOK /content/Rapid_Response
YOK /content/val2017
YOK /content/teknocan_full
YOK /content/bilgisayar_full
YOK /content/datasets/rapid_response

=== Drive dataset yapısı ===
dataset içi: dataset YOK
sofor_eylemi içi: sofor_eylemi YOK


In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os
print('dataset var mı:', os.path.isdir('/content/drive/MyDrive/dataset'))
if os.path.isdir('/content/drive/MyDrive/dataset'):
    print('içi:', os.listdir('/content/drive/MyDrive/dataset'))
print('model var mı:', os.path.exists('/content/drive/MyDrive/best_model_7class.pt'))

Mounted at /content/drive
dataset var mı: True
içi: ['Driver Distraction.v1i.yolov11.zip', 'Drowsiness Detection.v2-augmented-v1.yolov11.zip', 'augmentationn python', 'Detrac Videos.zip', 'sofor_eylemi', 'arac_tipi', 'arac_renk', 'plaka', 'hiz', 'nesne_yolcu', 'teknocan_full.zip', 'Cigarettes-reality-2.v21-mosaic.yolov11.zip', 'EĞİTİRKEN BU PAKETLERİ KLASÖRE İMPORT ET', 'archive (2).zip', 'labels_mapping.json', 'weights', 'best_model.pt', 'model_weights', 'coco']
model var mı: True


In [4]:
%cd /content
!git clone https://${GH_TOKEN}@github.com/f-tsakir/Rapid_Response.git
%cd Rapid_Response
!pip -q install ultralytics pyyaml

/content
Cloning into 'Rapid_Response'...
remote: Enumerating objects: 228, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 228 (delta 0), reused 2 (delta 0), pack-reused 218 (from 1)
Receiving objects: 100% (228/228), 17.12 MiB | 36.45 MiB/s, done.
Resolving deltas: 100% (83/83), done.
/content/Rapid_Response
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.1 MB/s eta 0:00:00


In [5]:
import os
d='/content/drive/MyDrive/dataset'
print('dataset kökü:', os.listdir(d))
print('nesne_yolcu :', os.listdir(d+'/nesne_yolcu') if os.path.isdir(d+'/nesne_yolcu') else 'YOK')
print('sofor_eylemi:', os.listdir(d+'/sofor_eylemi') if os.path.isdir(d+'/sofor_eylemi') else 'YOK')
print('plaka       :', os.listdir(d+'/plaka') if os.path.isdir(d+'/plaka') else 'YOK')

dataset kökü: ['Driver Distraction.v1i.yolov11.zip', 'Drowsiness Detection.v2-augmented-v1.yolov11.zip', 'augmentationn python', 'Detrac Videos.zip', 'sofor_eylemi', 'arac_tipi', 'arac_renk', 'plaka', 'hiz', 'nesne_yolcu', 'teknocan_full.zip', 'Cigarettes-reality-2.v21-mosaic.yolov11.zip', 'EĞİTİRKEN BU PAKETLERİ KLASÖRE İMPORT ET', 'archive (2).zip', 'labels_mapping.json', 'weights', 'best_model.pt', 'model_weights', 'coco']
nesne_yolcu : ['teknocan_full.zip', 'bilgisayar_full.zip']
sofor_eylemi: ['state_farm', 'abnormal_behaviour', 'driver_behaviors']
plaka       : ['tr_plates']


In [3]:
%cd /content
!rm -f annotations_trainval2017.zip annotations_trainval2017.zip.*
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -qo annotations_trainval2017.zip
print("--- kontrol ---")
import os
print('val2017 görsel:', len(os.listdir('/content/val2017')) if os.path.isdir('/content/val2017') else 'YOK')
print('instances json:', os.path.exists('/content/annotations/instances_val2017.json'))

/content
--2026-06-26 18:07:02--  http://images.cocodataset.org/annotations/annotations_trainval2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.246.106, 52.216.61.201, 16.15.213.101, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.246.106|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 252907541 (241M) [application/zip]
Saving to: ‘annotations_trainval2017.zip’

annotations_trainva 100%[===================>] 241.19M  51.2MB/s    in 5.0s    

2026-06-26 18:07:07 (48.1 MB/s) - ‘annotations_trainval2017.zip’ saved [252907541/252907541]

--- kontrol ---
val2017 görsel: 5000
instances json: True


In [4]:
import os, glob, yaml
for name in ['teknocan_full','bilgisayar_full','sigara_ds','seatbelt_ds']:
    b='/content/'+name; print('===',name,'===')
    print(' içerik:', os.listdir(b) if os.path.isdir(b) else 'YOK')
    dy=glob.glob(b+'/**/data.yaml', recursive=True)
    if dy: print(' classes:', yaml.safe_load(open(dy[0])).get('names'))
    for t in ['train/images','images/train','images']:
        if os.path.isdir(b+'/'+t): print(' img:', t, '->', len(os.listdir(b+'/'+t))); break

=== teknocan_full ===
 içerik: ['images', 'labels']
 img: images -> 108
=== bilgisayar_full ===
 içerik: ['images', 'labels']
 img: images -> 236
=== sigara_ds ===
 içerik: ['README.dataset.txt', 'data.yaml', 'README.roboflow.txt', 'valid', 'train']
 classes: ['0']
 img: train/images -> 3852
=== seatbelt_ds ===
 içerik: ['test', 'README.dataset.txt', 'data.yaml', 'README.roboflow.txt', 'valid', 'train']
 classes: ['Mobile phone', 'seat belt']
 img: train/images -> 285


In [5]:
import yaml
sources = [
  {'name':'coco_objects','format':'coco','images':'/content/val2017',
   'ann':'/content/annotations/instances_val2017.json',
   'map':{'laptop':'bilgisayar','cell phone':'telefon','bottle':'sise','cup':'sise',
          'person':'kisi','car':'arac','truck':'arac','bus':'arac'}},
  {'name':'teknocan_committee','format':'yolo','images':'/content/teknocan_full/images',
   'names':['arac','plaka','teknocan','bilgisayar','telefon','sigara','sise','emniyet_kemeri','kisi'],
   'map':{'arac':'arac','plaka':'plaka','teknocan':'teknocan','telefon':'telefon','kisi':'kisi'}},
  {'name':'bilgisayar_committee','format':'yolo','images':'/content/bilgisayar_full/images',
   'names':['arac','plaka','teknocan','bilgisayar','telefon','sigara','sise','emniyet_kemeri','kisi'],
   'map':{'arac':'arac','plaka':'plaka','teknocan':'teknocan','bilgisayar':'bilgisayar','telefon':'telefon','kisi':'kisi'}},
  {'name':'sigara_ds','format':'yolo','images':'/content/sigara_ds/train/images',
   'names':['0'],'map':{'0':'sigara'}},
  {'name':'seatbelt_ds','format':'yolo','images':'/content/seatbelt_ds/train/images',
   'names':['Mobile phone','seat belt'],'map':{'Mobile phone':'telefon','seat belt':'emniyet_kemeri'}},
]
yaml.safe_dump(sources, open('/content/Rapid_Response/training/sources.yaml','w'), allow_unicode=True)
print('✅ sources:', [s['name'] for s in sources])

✅ sources: ['coco_objects', 'teknocan_committee', 'bilgisayar_committee', 'sigara_ds', 'seatbelt_ds']


In [6]:
%cd /content/Rapid_Response
!python training/prepare_dataset.py --sources training/sources.yaml --data-yaml training/data.yaml \
  --out /content/datasets/rapid_response --split 0.70 0.15 0.15 --clean

/content/Rapid_Response

=== KAYNAK OZETI (goruntu) ===
  coco_objects                   3305  (2313/495/497)
  teknocan_committee              108  (75/16/17)
  bilgisayar_committee            236  (165/35/36)
  sigara_ds                      3850  (2695/577/578)
  seatbelt_ds                     285  (199/42/44)

=== SPLIT (goruntu) ===
  train  5447
  val    1165
  test   1172

=== SINIF HISTOGRAMI (kutu) ===
   0 arac                  2976
   1 plaka                  324
   2 teknocan               324
   3 bilgisayar             547
   4 telefon                377
   5 sigara               14717
   6 sise                  1924
   7 emniyet_kemeri         305
   8 kisi                 11454

Cikti: /content/datasets/rapid_response  (data.yaml: training/data.yaml)


In [8]:
import os, glob, shutil, random, sys, cv2
sys.path.insert(0, '/content/Rapid_Response/training')
from augment import TRANSFORMS, sample_params
img_dir='/content/datasets/rapid_response/images/train'
lbl_dir='/content/datasets/rapid_response/labels/train'
imgs=[f for f in glob.glob(img_dir+'/*') if f.lower().endswith(('.jpg','.png','.jpeg'))]
print(f"{len(imgs)} görsele sis/yağmur eklenecek...")
random.seed(0); added=0
for i, fp in enumerate(imgs):
    img=cv2.imread(fp)
    if img is None: continue
    h,w=img.shape[:2]
    t=random.choice(['fog','rain'])
    out=TRANSFORMS[t](img, **sample_params(t,h,w))
    base=os.path.splitext(os.path.basename(fp))[0]
    cv2.imwrite(os.path.join(img_dir,f"{base}_{t}.jpg"), out)
    lf=os.path.join(lbl_dir, base+'.txt')
    if os.path.exists(lf): shutil.copy(lf, os.path.join(lbl_dir,f"{base}_{t}.txt"))
    added+=1
    if (i+1)%500==0: print(f"  {i+1}/{len(imgs)} işlendi...")
for c in glob.glob(lbl_dir+'/*.cache'): os.remove(c)
print(f"✅ {added} sis/yağmur varyantı eklendi → train ~{len(imgs)+added}")

7515 görsele sis/yağmur eklenecek...
  500/7515 işlendi...


KeyboardInterrupt: 

In [ ]:
%cd /content/datasets
!zip -qr /content/drive/MyDrive/rapid_response_9class.zip rapid_response
import os
mb = os.path.getsize('/content/drive/MyDrive/rapid_response_9class.zip')//1048576
print(f'✅ dataset kaydedildi: {mb} MB → MyDrive/rapid_response_9class.zip')

/content/datasets


In [11]:
%cd /content/Rapid_Response
!ls /content/datasets/rapid_response
!ls /content/datasets/rapid_response/images
print('--- data.yaml ---')
!grep -E "path|train|val|test|nc" training/data.yaml

/content/Rapid_Response
images	labels
test  train  val
--- data.yaml ---
# (VeRi-776 crop'lari, train_tip.py) ogrenilir; gercek-sahne YOLO'su kirlenmez.
#   - arac tipi:   VeRi-776 crop'lari -> train_tip.py (YOLO'da DEGIL)
#   - teknocan:    YARISMAYA OZEL -> kendi (sari oyuncak) + copy_paste
# train/val/test orani onerisi: 0.70 / 0.15 / 0.15 (dengeli, sinif bazli).
path: ../datasets/rapid_response   # veri seti kok dizini (gelistirme makinesi)
train: images/train
val: images/val
test: images/test
nc: 9


## 7) Detektörü eğit (T4/A100)
Parametreler `config.yaml > training:`'ten okunur. Bitince `weights/best_model.pt` üretir.


In [ ]:
!python training/train_yolo.py --config config/config.yaml


## 8) Sonuçları indir
best_model.pt'yi indir. color_cnn.pt + tip_cnn.pt'yi handoff paketinden alıp image'a göm.


In [ ]:
from google.colab import files
files.download('weights/best_model.pt')


---
**Sonraki:** `best_model.pt` + `color_cnn.pt` + `tip_cnn.pt` → Dockerfile `weights/`'e.
EasyOCR + MediaPipe için: `python training/fetch_assets.py`.
